# LA LA Stay · Airbnb Los Ángeles
### Análisis de datos — Andrés Arismendi (Uniandes)

**Fuente:** Inside Airbnb, `listings.csv.gz` de Los Ángeles (extracción del 14 al 16 de septiembre de 2026), guardado en `data/raw/`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)

In [ ]:
# Rutas del proyecto: funciona desde cualquier carpeta dentro del repositorio
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / 'data').is_dir():
            return folder
    return start

ROOT = find_project_root()
RAW_FILE = ROOT / 'data' / 'raw' / 'listings.csv.gz'
print('Archivo:', RAW_FILE)

### Cargar los datos
`id` y `host_id` se leen como **texto**: los IDs de Inside Airbnb tienen hasta 19 dígitos y, si se leen como número o pasan por Excel, pierden los últimos dígitos.

In [ ]:
df = pd.read_csv(RAW_FILE, compression='gzip', low_memory=False,
                 dtype={'id': str, 'host_id': str})

print(f'Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}')
df.head()

## 1 · Pasar las columnas JSON a columnas
Dos columnas guardan texto en formato JSON: `price_quote_raw` (objeto) y `amenities` (lista). Primero lo confirmamos.

In [ ]:
import json

def looks_like_json(series, sample=500):
    values = series.dropna().astype(str).head(sample)
    if values.empty or not values.str.strip().str[:1].isin(['{', '[']).all():
        return False
    try:
        values.map(json.loads)
        return True
    except (ValueError, TypeError):
        return False

json_columns = [col for col in df.select_dtypes('object') if looks_like_json(df[col])]
print('Columnas JSON:', json_columns)
print('Shape inicial:', df.shape)

### 1.1 · `price_quote_raw`: cada llave es una columna
Cada fila trae `{"quote": {...}}` con 17 llaves. Cada llave pasa a una columna con el prefijo `quote_`.

La llave `raw_price_line_items` es a su vez una **lista** de cobros (hasta 12 por anuncio), y cada cobro tiene 4 llaves: `amount`, `item_type`, `description` y `price_string`. Se expande por posición: `quote_line_item_1_amount`, `quote_line_item_1_item_type`, …, `quote_line_item_12_price_string`. Si un anuncio tiene menos cobros, las columnas sobrantes quedan vacías.

In [ ]:
quotes = df['price_quote_raw'].map(lambda raw: json.loads(raw)['quote'] if pd.notna(raw) else {})
quote_df = pd.json_normalize(quotes.tolist()).add_prefix('quote_').set_index(df.index)

# Expandir la lista de cobros por posición
line_items = quote_df.pop('quote_raw_price_line_items')
item_keys = ['amount', 'item_type', 'description', 'price_string']
max_items = int(line_items.map(lambda x: len(x) if isinstance(x, list) else 0).max())

items_df = pd.DataFrame(
    [[(items[i].get(key) if isinstance(items, list) and i < len(items) else None)
      for i in range(max_items) for key in item_keys]
     for items in line_items],
    columns=[f'quote_line_item_{i + 1}_{key}' for i in range(max_items) for key in item_keys],
    index=df.index,
)

print('Llaves de quote:', quote_df.shape[1] + 1, '(incluye raw_price_line_items)')
print('Máximo de cobros por anuncio:', max_items, '->', items_df.shape[1], 'columnas de cobros')

In [ ]:
# Tipos de dato: montos a número, fechas a fecha, banderas a booleano
money = ['quote_taxes', 'quote_total_price', 'quote_discount_amount', 'quote_price_per_night',
         'quote_nightly_subtotal', 'quote_discounted_subtotal']
quote_df[money] = quote_df[money].apply(pd.to_numeric, errors='coerce')
date_cols = [c for c in quote_df.columns if c.endswith('_date')]
quote_df[date_cols] = quote_df[date_cols].apply(pd.to_datetime)
for col in ['quote_is_available', 'quote_is_monthly_priced']:
    quote_df[col] = quote_df[col].astype('boolean')

amount_cols = [c for c in items_df.columns if c.endswith('_amount')]
items_df[amount_cols] = items_df[amount_cols].apply(pd.to_numeric, errors='coerce')

df = pd.concat([df.drop(columns=['price_quote_raw']), quote_df, items_df], axis=1)
print('Shape después de price_quote_raw:', df.shape)

In [ ]:
# Llaves que vienen vacías en TODAS las filas (se conservan, pero conviene saberlo)
empty_quote = [c for c in quote_df.columns if quote_df[c].isna().all()]
print('Llaves de quote siempre vacías:', empty_quote)

### 1.2 · `amenities`
`amenities` es una **lista de textos, sin llaves** (por ejemplo `["Wifi", "Kitchen", "Pool"]`). Pasarla a columnas significa crear **una columna por servicio** (1 si el anuncio lo tiene, 0 si no). Aquí solo se calcula cuántas columnas resultarían, sin crearlas todavía.

In [ ]:
amenities_list = df['amenities'].map(lambda raw: [a.strip().lower() for a in json.loads(raw)] if pd.notna(raw) else [])
n_amenities = amenities_list.explode().nunique()
print(f'Servicios distintos: {n_amenities:,}')
print(f'Shape si cada servicio fuera una columna: ({df.shape[0]:,}, {df.shape[1] - 1 + n_amenities:,})')

### 1.3 · ¿Qué valores toma `amenities`?
Tabla de frecuencia de **todos** los servicios: en cuántos anuncios aparece cada uno y qué porcentaje del total representa.

In [ ]:
amenity_freq = (amenities_list.explode().dropna()
                .value_counts()
                .rename_axis('amenity')
                .reset_index(name='anuncios'))
amenity_freq['% anuncios'] = (amenity_freq['anuncios'] / len(df) * 100).round(2)

print(f'Servicios distintos: {len(amenity_freq):,}')
print(f'Anuncios sin ningún servicio: {(amenities_list.str.len() == 0).sum()}')
amenity_freq.head(60)

**Distribución por frecuencia:** cuántos servicios hay en cada rango de popularidad.

In [ ]:
n = len(df)
bands = pd.cut(amenity_freq['anuncios'],
               bins=[0, 1, 0.001 * n, 0.01 * n, 0.05 * n, 0.90 * n, n],
               labels=['1 solo anuncio', '2 anuncios a 0,1 %', '0,1 % – 1 %',
                       '1 % – 5 %', '5 % – 90 %', 'más de 90 %'])
band_table = bands.value_counts().sort_index().to_frame('servicios')
band_table['% de servicios'] = (band_table['servicios'] / len(amenity_freq) * 100).round(1)
band_table

**Variantes del mismo servicio:** muchos servicios son el mismo con distinta marca, tamaño o detalle. Ejemplos por palabra clave:

In [ ]:
for keyword in ['tv', 'wifi', 'shampoo', 'pool', 'parking', 'coffee']:
    variants = amenity_freq[amenity_freq['amenity'].str.contains(keyword, regex=False)]
    print(f"'{keyword}': {len(variants):,} variantes -> {variants['amenity'].head(6).tolist()}")

print('\nEjemplos de servicios que aparecen en un solo anuncio:')
print(amenity_freq.loc[amenity_freq['anuncios'] == 1, 'amenity'].sample(8, random_state=1).tolist())

### 1.4 · Unificar las variantes de `amenities`
Cada servicio unificado se define con una **expresión regular** sobre el texto en minúsculas. Un anuncio tiene el servicio si **alguno** de sus textos coincide. Ejemplos:
- `tv` agrupa "tv", "hdtv", "55 inch hdtv with netflix"…
- `pool` agrupa "pool", "private pool", "shared outdoor pool…", pero **no** "pool table" ni "pool view".
- `washer` excluye "dishwasher"; `dryer` excluye "hair dryer".

In [ ]:
import re

AMENITY_RULES = {
    # Básicos
    'wifi':              r'\bwi-?fi\b',
    'kitchen':           r'\bkitchen\b',
    'tv':                r'\b(?:hd)?tv\b',
    'workspace':         r'workspace',
    'coffee_maker':      r'coffee maker|espresso|nespresso|keurig',
    # Clima
    'air_conditioning':  r'air conditioning|\bac\b|a/c',
    'heating':           r'heating|\bheater\b',
    'fireplace':         r'fireplace',
    'ceiling_fan':       r'ceiling fan',
    # Lavado y baño
    'washer':            r'(?<!dish)washer',
    'dryer':             r'(?<!hair )\bdryer\b',
    'dishwasher':        r'dishwasher',
    'bathtub':           r'bathtub|soaking tub',
    # Exterior
    'pool':              r'\bpool\b(?! table)(?! view)',
    'hot_tub':           r'hot tub|jacuzzi',
    'patio_balcony':     r'patio|balcony',
    'backyard':          r'backyard',
    'bbq_grill':         r'\bbbq\b|barbecue|\bgrill\b',
    'outdoor_dining':    r'outdoor dining|outdoor furniture',
    'fire_pit':          r'fire pit',
    'garden':            r'\bgarden\b',
    # Acceso y estacionamiento
    'free_parking':      r'free .*parking',
    'paid_parking':      r'paid .*parking',
    'ev_charger':        r'ev charger',
    'private_entrance':  r'private entrance',
    'self_check_in':     r'self check-?in|keypad|lockbox|smart lock',
    'elevator':          r'\belevator\b',
    # Ocio
    'gym':               r'\bgym\b|exercise equipment',
    'sound_system':      r'sound system|bluetooth speaker',
    'game_console':      r'game console|playstation|\bps[45]\b|xbox|nintendo',
    # Ubicación / vistas
    'beach_access':      r'beach access|beachfront',
    'ocean_view':        r'ocean view|sea view|beach view',
    'mountain_view':     r'mountain view',
    'city_view':         r'city skyline view|city view',
    # Familias y políticas
    'crib':              r'\bcrib\b|pack .n play',
    'pets_allowed':      r'pets allowed',
    'long_term_stays':   r'long term stays allowed',
}

joined = amenities_list.map(' | '.join)          # todos los servicios del anuncio en un texto
for name, pattern in AMENITY_RULES.items():
    df[f'am_{name}'] = joined.str.contains(pattern, regex=True).astype(int)
df['amenities_count'] = amenities_list.str.len()

am_cols = [f'am_{name}' for name in AMENITY_RULES]
print(f'{len(am_cols)} servicios unificados creados | Shape: {df.shape}')

In [ ]:
# Verificación: qué textos originales quedaron dentro de algunos servicios unificados
exploded = amenity_freq.copy()
for name in ['tv', 'pool', 'washer', 'dryer', 'free_parking']:
    hits = exploded[exploded['amenity'].str.contains(AMENITY_RULES[name], regex=True)]
    print(f"{name}: {len(hits):,} variantes, p. ej. {hits['amenity'].head(4).tolist()}")

### 1.5 · Frecuencia y relación con el precio
Para comparar precios usamos el **mercado objetivo del Charter**: hosts con 1–2 anuncios, con precio y **sin precio mensual** (`quote_is_monthly_priced = False`).
Se compara el **precio mediano** de los anuncios con y sin cada servicio. Es una comparación **descriptiva, no causal**: un servicio puede ir junto con propiedades más grandes o mejor ubicadas.

In [ ]:
df['price_num'] = pd.to_numeric(df['price'].str.replace(r'[\$,]', '', regex=True), errors='coerce')
target = df[df['calculated_host_listings_count'].between(1, 2)
            & df['price_num'].notna()
            & (df['quote_is_monthly_priced'] == False)]
print(f'Anuncios del mercado objetivo: {len(target):,}')

rows = []
for col in am_cols:
    has = target[col] == 1
    rows.append({
        'servicio': col.removeprefix('am_'),
        '% anuncios (todos)': df[col].mean() * 100,
        '% mercado objetivo': has.mean() * 100,
        'mediana con': target.loc[has, 'price_num'].median(),
        'mediana sin': target.loc[~has, 'price_num'].median(),
    })
amenity_summary = pd.DataFrame(rows)
amenity_summary['diferencia %'] = (amenity_summary['mediana con'] / amenity_summary['mediana sin'] - 1) * 100
amenity_summary['entre 5 % y 90 %'] = amenity_summary['% mercado objetivo'].between(5, 90)
amenity_summary = amenity_summary.sort_values('% mercado objetivo', ascending=False).round(1).reset_index(drop=True)
amenity_summary

### 1.6 · Servicios seleccionados
Se descartan 9 servicios unificados:
- **Casi universales (más del 90 % del mercado objetivo):** `wifi`, `tv`, `free_parking`. No diferencian un anuncio de otro.
- **Muy raros (menos del 5 %):** `ocean_view`, `game_console`.
- **Diferencia de precio pequeña o ambigua:** `ceiling_fan`, `elevator`, `private_entrance` (5 %–8 %) y `paid_parking` (negativa: suele indicar zonas densas, no un servicio que agregue valor).

Se conservan los demás servicios más `amenities_count`, y se elimina la columna JSON original `amenities`.

In [ ]:
DROP_AMENITIES = ['wifi', 'tv', 'free_parking', 'ocean_view', 'game_console',
                  'ceiling_fan', 'elevator', 'private_entrance', 'paid_parking']

df = df.drop(columns=[f'am_{name}' for name in DROP_AMENITIES] + ['amenities'])
am_cols = [col for col in df.columns if col.startswith('am_')]

print(f'Servicios conservados ({len(am_cols)}):')
print([col.removeprefix('am_') for col in am_cols])
print(f'\nShape: {df.shape}')

## 2 · Selección de columnas
### 2.1 · Variables derivadas antes de eliminar
Antes de quitar las columnas de la cotización se calculan dos variables útiles:
- `quote_nights`: noches que se cotizaron (de 1 a 365, según las noches mínimas).
- `base_nightly_rate`: **tarifa publicada por noche sin descuentos** = `quote_nightly_subtotal / quote_nights`. Existe para los anuncios con precio por noche (no mensuales). Es una posible variable objetivo alternativa a `price`, que ya trae descuentos aplicados.

In [ ]:
df['quote_nights'] = (df['quote_requested_checkout_date'] - df['quote_requested_checkin_date']).dt.days
df['base_nightly_rate'] = df['quote_nightly_subtotal'] / df['quote_nights']

print(f"Anuncios con tarifa base: {df['base_nightly_rate'].notna().sum():,}")
print(pd.crosstab(df['base_nightly_rate'].notna(), df['quote_is_monthly_priced'],
                  rownames=['tiene tarifa base'], colnames=['precio mensual']))

### 2.2 · Columnas a eliminar, por grupo
Cada grupo indica **por qué** se elimina. Al final se verifica que ninguna columna quede sin clasificar.

In [ ]:
always_empty = [col for col in df.columns if df[col].isna().all()]

DROP_GROUPS = {
    'Vacías en todas las filas': always_empty,
    'Metadatos de la extracción': [
        'scrape_id', 'last_scraped', 'calendar_last_scraped', 'source', 'quote_currency',
        'quote_requested_checkin_date', 'quote_requested_checkout_date'],
    'URLs, identificadores secundarios y texto libre': [
        'listing_url', 'picture_url', 'host_id', 'host_url', 'host_profile_id', 'host_profile_url',
        'host_picture_url', 'host_name', 'name', 'description', 'host_about', 'host_location', 'license'],
    'Derivadas del precio (fuga hacia la variable objetivo)': [
        'price', 'price_quote_checkin_date', 'price_quote_checkout_date', 'price_quote_total_price',
        'price_quote_price_per_night', 'estimated_revenue_l365d', 'quote_total_price',
        'quote_price_per_night', 'quote_taxes', 'quote_discount_amount', 'quote_nightly_subtotal',
        'quote_discounted_subtotal'] + [c for c in df.columns if c.startswith('quote_line_item_')],
    'Redundantes': [
        'minimum_minimum_nights', 'maximum_minimum_nights', 'minimum_maximum_nights',
        'maximum_maximum_nights', 'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm',
        'host_listings_count', 'calculated_host_listings_count_entire_homes',
        'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms',
        'hosts_time_as_user_years', 'hosts_time_as_user_months'],
    'Desempeño del anuncio (un anfitrión nuevo no lo conoce)': [
        'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness',
        'review_scores_checkin', 'review_scores_communication', 'review_scores_location',
        'review_scores_value', 'number_of_reviews_l30d', 'number_of_reviews_ly', 'reviews_per_month',
        'first_review', 'availability_30', 'availability_60', 'availability_90', 'availability_eoy',
        'host_is_superhost', 'host_has_profile_pic', 'host_identity_verified',
        'hosts_time_as_host_years', 'hosts_time_as_host_months', 'quote_is_available'],
}

# Una columna puede estar en "vacías" y en otro grupo; se cuenta una sola vez
to_drop = list(dict.fromkeys(col for cols in DROP_GROUPS.values() for col in cols))
missing = [col for col in to_drop if col not in df.columns]
assert not missing, f'Columnas inexistentes: {missing}'

summary = pd.DataFrame({'grupo': list(DROP_GROUPS),
                        'columnas': [len(cols) for cols in DROP_GROUPS.values()]})
display(summary)
print(f'Total a eliminar (sin repetir): {len(to_drop)}')

In [ ]:
df = df.drop(columns=to_drop)
print(f'Shape: {df.shape}')

### 2.3 · Columnas conservadas

In [ ]:
KEEP_GROUPS = {
    'Clave': ['id'],
    'Variables objetivo (candidatas)': ['price_num', 'base_nightly_rate'],
    'Características de la propiedad': [
        'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude',
        'property_type', 'room_type', 'accommodates', 'bedrooms', 'beds', 'bathrooms_text'],
    'Servicios': [c for c in df.columns if c.startswith('am_')] + ['amenities_count'],
    'Filtros del mercado objetivo': [
        'calculated_host_listings_count', 'minimum_nights', 'maximum_nights',
        'quote_is_monthly_priced', 'quote_nights'],
    'Actividad (solo análisis, no van al modelo)': [
        'number_of_reviews', 'number_of_reviews_ltm', 'last_review', 'availability_365',
        'estimated_occupancy_l365d', 'has_availability'],
}
classified = [col for cols in KEEP_GROUPS.values() for col in cols]
unclassified = sorted(set(df.columns) - set(classified))
assert not unclassified, f'Columnas sin clasificar: {unclassified}'

df = df[classified]
df['last_review'] = pd.to_datetime(df['last_review'])
df['has_availability'] = df['has_availability'].map({'t': True, 'f': False}).astype('boolean')

display(pd.DataFrame({'grupo': list(KEEP_GROUPS), 'columnas': [len(c) for c in KEEP_GROUPS.values()]}))
print(f'Shape final: {df.shape}')